In [1]:
import sqlite3, pandas as pd
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
CLEAN, SYN = ROOT / "data" / "clean", ROOT / "data" / "synthetic"
DB = ROOT / "data" / "upi.db"

tables = {
    "fact_txn":   SYN / "upi_synthetic_txns.csv",
    "dim_date":   CLEAN / "dim_date.csv",
    "dim_bank":   CLEAN / "dim_bank.csv",
    "remitter":   CLEAN / "remitter_clean.csv",
    "chargeback": CLEAN / "chargeback_clean.csv",
    "national":   CLEAN / "national_monthly.csv",
}
con = sqlite3.connect(DB)
for name, path in tables.items():
    df = pd.read_csv(path, low_memory=False)
    if name == "fact_txn":                       # empty text becomes NaN in CSV, so put "" back
        for c in ["festival_name", "failure_type", "failure_reason"]:
            df[c] = df[c].fillna("")
    df.to_sql(name, con, if_exists="replace", index=False)
    print(f"{name:11s} {len(df):>8,} rows")

con.executescript("""
CREATE INDEX IF NOT EXISTS ix_fact_date  ON fact_txn(date);
CREATE INDEX IF NOT EXISTS ix_fact_month ON fact_txn(month);
CREATE INDEX IF NOT EXISTS ix_fact_bank  ON fact_txn(remitter_bank);
""")
con.commit()

def q(sql):                                      # chhota helper: SQL do, table wapas lo
    return pd.read_sql(sql, con)

print(q("SELECT COUNT(*) AS rows, ROUND(100.0*AVG(status='Failed'),2) AS fail_pct FROM fact_txn"))

fact_txn     400,000 rows
dim_date         365 rows
dim_bank          66 rows
remitter         598 rows
chargeback     3,930 rows
national          30 rows
     rows  fail_pct
0  400000      9.67


In [2]:
q1 = q("""
WITH m AS (
  SELECT month, COUNT(*) AS txns,
         ROUND(100.0*AVG(status='Failed'),2) AS fail_pct,
         ROUND(100.0*AVG(failure_type='Technical'),3) AS tech_pct
  FROM fact_txn GROUP BY month)
SELECT month, txns, fail_pct,
       ROUND(fail_pct - LAG(fail_pct) OVER (ORDER BY month),2) AS change_vs_prev,
       tech_pct
FROM m ORDER BY month
""")
q1

,month,txns,fail_pct,change_vs_prev,tech_pct
0,2025-09,29066,7.86,NaN,0.347
1,2025-10,30676,7.87,0.01,0.368
2,2025-11,30457,8.22,0.35,0.355
3,2025-12,32403,9.23,1.01,0.444
4,2026-01,32539,9.47,0.24,0.237
5,2026-02,30650,9.30,-0.17,0.258
6,2026-03,34601,10.00,0.70,0.231
7,2026-04,34268,10.12,0.12,0.283
8,2026-05,35715,10.52,0.40,0.286
9,2026-06,35017,10.33,-0.19,0.343


In [3]:
q2 = q("""
SELECT hour, COUNT(*) AS txns,
       ROUND(100.0*AVG(failure_type='Technical'),3) AS tech_fail_pct,
       ROUND(100.0*AVG(failure_type='Business'),2)  AS biz_fail_pct
FROM fact_txn GROUP BY hour ORDER BY hour
""")

q3 = q("""
SELECT CASE WHEN is_festival=1 THEN 'Festival day' ELSE 'Normal day' END AS day_type,
       COUNT(*) AS txns,
       ROUND(100.0*AVG(status='Failed'),2) AS fail_pct,
       ROUND(100.0*AVG(failure_type='Technical'),3) AS tech_pct
FROM fact_txn GROUP BY is_festival
UNION ALL
SELECT CASE WHEN is_month_end=1 THEN 'Month-end day' ELSE 'Other days' END,
       COUNT(*), ROUND(100.0*AVG(status='Failed'),2), ROUND(100.0*AVG(failure_type='Technical'),3)
FROM fact_txn GROUP BY is_month_end
""")
display(q2); display(q3)

,hour,txns,tech_fail_pct,biz_fail_pct
0,0,5481,0.401,10.97
1,1,3617,0.332,10.40
2,2,2761,0.362,9.96
3,3,2169,0.461,10.60
4,4,1958,0.358,11.75
5,5,2714,0.405,10.65
6,6,5685,0.211,9.50
7,7,9150,0.361,9.52
8,8,13331,0.293,9.44
9,9,16814,0.280,9.26


,day_type,txns,fail_pct,tech_pct
0,Normal day,374134,9.71,0.297
1,Festival day,25866,9.04,0.545
2,Other days,371661,9.51,0.311
3,Month-end day,28339,11.80,0.332


In [4]:
q4 = q("""
WITH r AS (
  SELECT failure_type, failure_reason, COUNT(*) AS fails
  FROM fact_txn WHERE status='Failed' GROUP BY failure_type, failure_reason)
SELECT failure_type, failure_reason, fails,
       ROUND(100.0*fails/SUM(fails) OVER (),1) AS pct_of_all_failures,
       ROUND(100.0*SUM(fails) OVER (ORDER BY fails DESC)/SUM(fails) OVER (),1) AS cumulative_pct
FROM r ORDER BY fails DESC
""")

q5 = q("""
SELECT failure_type, failure_reason, COUNT(*) AS fails,
       ROUND(100.0*AVG(final_outcome='Recovered on retry'),1) AS recovered_pct,
       ROUND(100.0*AVG(final_outcome='Lost'),1) AS lost_pct
FROM fact_txn WHERE status='Failed'
GROUP BY failure_type, failure_reason ORDER BY fails DESC
""")
display(q4); display(q5)

,failure_type,failure_reason,fails,pct_of_all_failures,cumulative_pct
0,Business,Insufficient funds,17234,44.6,44.6
1,Business,Wrong UPI PIN,8883,23.0,67.5
2,Business,Invalid / inactive beneficiary,4537,11.7,79.3
3,Business,Daily limit exceeded,3766,9.7,89.0
4,Business,Declined by bank (risk / other),3006,7.8,96.8
5,Technical,Remitter bank server down / timeout,516,1.3,98.1
6,Technical,Network connectivity error,267,0.7,98.8
7,Technical,PSP / app technical error,241,0.6,99.4
8,Technical,NPCI / UPI switch timeout,227,0.6,100.0


,failure_type,failure_reason,fails,recovered_pct,lost_pct
0,Business,Insufficient funds,17234,11.9,88.1
1,Business,Wrong UPI PIN,8883,67.9,32.1
2,Business,Invalid / inactive beneficiary,4537,23.1,76.9
3,Business,Daily limit exceeded,3766,7.3,92.7
4,Business,Declined by bank (risk / other),3006,18.2,81.8
5,Technical,Remitter bank server down / timeout,516,55.8,44.2
6,Technical,Network connectivity error,267,56.6,43.4
7,Technical,PSP / app technical error,241,47.7,52.3
8,Technical,NPCI / UPI switch timeout,227,62.6,37.4


In [5]:
q6 = q("""
SELECT COALESCE(b.bank_type,'Grouped small banks (OTHER BANKS)') AS bank_type,
       COUNT(*) AS txns,
       ROUND(100.0*AVG(f.status='Failed'),2) AS fail_pct,
       ROUND(100.0*AVG(f.failure_type='Technical'),3) AS tech_pct
FROM fact_txn f LEFT JOIN dim_bank b ON f.remitter_bank = b.bank_key
GROUP BY 1 ORDER BY txns DESC
""")

q7 = q("""
SELECT amount_bucket, COUNT(*) AS txns, ROUND(100.0*AVG(status='Failed'),2) AS fail_pct
FROM fact_txn GROUP BY amount_bucket
ORDER BY CASE amount_bucket WHEN '<100' THEN 1 WHEN '100-499' THEN 2 WHEN '500-999' THEN 3
         WHEN '1k-4.9k' THEN 4 WHEN '5k-9.9k' THEN 5 ELSE 6 END
""")

q8 = q("""
SELECT network, COUNT(*) AS txns, ROUND(100.0*AVG(failure_type='Technical'),3) AS tech_pct
FROM fact_txn GROUP BY network ORDER BY tech_pct DESC
""")
display(q6); display(q7); display(q8)

,bank_type,txns,fail_pct,tech_pct
0,Public sector bank,232374,9.23,0.317
1,Private sector bank,77267,6.74,0.060
2,Grouped small banks (OTHER BANKS),64009,10.56,0.569
3,Payments bank,26350,19.96,0.398


,amount_bucket,txns,fail_pct
0,<100,21095,8.01
1,100-499,149266,8.42
2,500-999,81853,9.46
3,1k-4.9k,130359,10.81
4,5k-9.9k,14635,13.86
5,10k+,2792,19.91


,network,txns,tech_pct
0,3G,19929,0.502
1,4G,240012,0.312
2,WiFi,40308,0.298
3,5G,99751,0.284


In [6]:
q9 = q("""
WITH bank_fail AS (
  SELECT remitter_bank, COUNT(*) AS txns, 100.0*AVG(status='Failed') AS fail_pct
  FROM fact_txn WHERE remitter_bank <> 'OTHER BANKS' GROUP BY remitter_bank),
ranked AS (SELECT *, NTILE(4) OVER (ORDER BY fail_pct) AS quartile FROM bank_fail)
SELECT quartile, COUNT(*) AS banks, ROUND(AVG(fail_pct),2) AS avg_fail_pct,
       ROUND(MIN(fail_pct),2) AS min_fail, ROUND(MAX(fail_pct),2) AS max_fail
FROM ranked GROUP BY quartile ORDER BY quartile
""")

q10 = q("""
SELECT month, SUM(total_txns) AS txns, SUM(cb_received) AS chargebacks,
       ROUND(1e6*SUM(cb_received)/SUM(total_txns),2) AS cb_per_million
FROM chargeback
WHERE month >= '2025-11' AND is_tiny_bank = 0
GROUP BY month ORDER BY month
""")
display(q9); display(q10)

,quartile,banks,avg_fail_pct,min_fail,max_fail
0,1,4,6.23,4.62,8.14
1,2,4,8.86,8.34,9.27
2,3,4,10.62,9.84,11.44
3,4,3,17.57,12.49,22.56


,month,txns,chargebacks,cb_per_million
0,2025-11,20435067751,161833,7.92
1,2025-12,21601580778,181711,8.41
2,2026-01,21671045096,185879,8.58
3,2026-02,20372330724,168901,8.29
4,2026-03,22912880883,181877,7.94
5,2026-04,22606676831,210304,9.30
6,2026-05,23475005617,188867,8.05
7,2026-06,22974479838,193698,8.43
8,2026-07,23955574395,199408,8.32
